# Tarea Unidad 3

## Milena Fernanda Rivera Hernández

#Clasificación de secuencias de aminoácidos

En la carpeta Datos/TareaUnidad_3

Ustedes encontraran cuatro carpetas, Cada carpeta contiene dos archivos .txt, con secuencias de aminoácidos en formato Fasta, uno de los archivos inicia con "non-*.txt" que indica que son secuencias que codifican a proteínas diferentes a las que queremos identificar (es decir, es el conjunto negativo) y el otro archivo contiene secuencias de proteínas de la misma familia.

El objetivo del trabajo es realizar un clasificador, que permita identificar si una proteína es o no de la familia que estamos estudian.

Paso:

Seleccione uno de los cuatro conjuntos de datos

Lea cada archivo de manera independiente linea a linea, y construya un dataframe, donde concatene los dos archivos, que contenga tres columnas

Columna 1: El identificar iniciando por el símbolo >

columna 2: La secuencias

Columna 3: La etiqueta

Es importante que justifique la codificación de la secuencias.

In [4]:
import sys
import numpy as np
import sklearn
import pandas as pd
import re
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import zip_longest

import warnings
warnings.filterwarnings("ignore")

In [5]:
!pip install tensorflow

In [6]:
# Lea cada archivo de manera independiente linea a linea, y construya un
# dataframe, donde concatene los dos archivos, que contenga tres columnas

#Columna 1: El identificar iniciando por el símbolo >
#columna 2: La secuencias
#Columna 3: La etiqueta

def constuir_dataframe(file_r, signo):
    """
    Lee un archivo de secuencias y extrae identificadores, secuencias y
    etiquetas
    """
    data = []
    current_id = ""
    current_seq = ""

    with open(file_r, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith('>'):
                # Si ya teníamos una secuencia previa, la guardamos antes de procesar el nuevo ID
                if current_id:
                    # Extraer etiqueta (lo que sigue al último |)
                    label = current_id.split('|')[-1]
                    if signo == '+':
                      classs = '+'
                      data.append([classs, current_id, current_seq, label])
                    else:
                      data.append([signo, current_id, current_seq, label])
                    current_seq = ""

                current_id = line
            else:
                current_seq += line

        # Aniadir ultima secuencia del archivo
        if current_id:
            label = current_id.split('|')[-1]
            if signo == '+':
              classs = '+'
              data.append([classs, current_id, current_seq, label])
            else:
              classs = '-'
              data.append([classs, current_id, current_seq, label])

    return pd.DataFrame(data, columns=['Class','ID', 'Sequence', 'Label'])

file_hbp = '/content/hbp.txt'
file_nonhbp = '/content/non-hbp.txt'

df1 = constuir_dataframe(file_hbp, '+')
df2 = constuir_dataframe(file_nonhbp,'-')

# Combinar ambos DataFrames
combined_df = pd.concat([df1, df2], ignore_index=True)

# Mostrar el resultado
combined_df.head()

,Class,ID,Sequence,Label
0,+,>hormone|Q802V6|ABH2,MNTHESEVYTVAPEMPAMFDGMKLAAVATVLYVIVRCLNLKSPTAP...,ABH2
1,+,>hormone|P33487|ABP1,MIVLSVGSASSSPIVVVFSVALLLFYFSETSLGAPCPINGLPIVRN...,ABP1
2,+,>hormone|P33488|ABP4,MVRRRPATGAAPRPHLAAVGRGLLLASVLAAAASSLPVAESSCPRD...,ABP4
3,+,>hormone|P05067|A4_H,MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMN...,A4_H
4,+,>hormone|Q8BQS5|ADR2,MNEPAKHRLGCTRTPEPDIRLRKGHQLDDTRGSNNDNYQGDLEPSL...,ADR2


In [7]:
#Building our Dataset by creating a custom Pandas Dataframe
# Each column in a Dataframe is called a Series.
classes = combined_df.loc[:,"Class"]
print(classes)

0      +
1      +
2      +
3      +
4      +
      ..
241    -
242    -
243    -
244    -
245    -
Name: Class, Length: 246, dtype: object


In [8]:
# generate list of DNA sequence
sequences = list(combined_df.loc[: , "Sequence"])
dataset = {}

# loop hrough sequences and split into individual nucleotides
for i, seq in enumerate(sequences):

  # split into nucleotides, remove ta characters
  nucleotides = list(seq)
  nucleotides = [x for x in nucleotides if x != '\t']

  # append class assignment
  nucleotides.append(classes[i])

  # add to dataset
  dataset[i] = nucleotides

print(dataset[0])

['M', 'N', 'T', 'H', 'E', 'S', 'E', 'V', 'Y', 'T', 'V', 'A', 'P', 'E', 'M', 'P', 'A', 'M', 'F', 'D', 'G', 'M', 'K', 'L', 'A', 'A', 'V', 'A', 'T', 'V', 'L', 'Y', 'V', 'I', 'V', 'R', 'C', 'L', 'N', 'L', 'K', 'S', 'P', 'T', 'A', 'P', 'P', 'D', 'L', 'T', 'F', 'Q', 'D', 'T', 'T', 'L', 'N', 'H', 'F', 'L', 'L', 'K', 'S', 'C', 'P', 'I', 'L', 'T', 'K', 'E', 'Y', 'I', 'P', 'P', 'L', 'L', 'W', 'G', 'K', 'S', 'G', 'H', 'L', 'Q', 'T', 'A', 'L', 'Y', 'G', 'K', 'L', 'G', 'R', 'V', 'S', 'S', 'P', 'H', 'P', 'F', 'G', 'L', 'R', 'K', 'Y', 'L', 'P', 'M', 'Q', 'D', 'G', 'A', 'T', 'A', 'T', 'F', 'D', 'L', 'F', 'E', 'P', 'L', 'A', 'D', 'H', 'Q', 'S', 'G', 'E', 'D', 'V', 'T', 'M', 'V', 'I', 'C', 'P', 'G', 'I', 'G', 'N', 'H', 'S', 'E', 'K', 'H', 'Y', 'I', 'R', 'T', 'F', 'V', 'D', 'H', 'S', 'Q', 'K', 'Q', 'G', 'Y', 'R', 'C', 'A', 'V', 'L', 'N', 'H', 'L', 'G', 'A', 'L', 'P', 'N', 'I', 'E', 'L', 'T', 'S', 'P', 'R', 'M', 'F', 'T', 'Y', 'G', 'C', 'T', 'W', 'E', 'F', 'A', 'A', 'M', 'V', 'G', 'F', 'I', 'K', 'K', 'T',

In [9]:
# Obtener secuencias y clases
#sequences = combined_df['Sequence'].tolist()
#classes = combined_df['Class'].tolist()

# Encontrar la longitud máxima de secuencia
max_length = max(len(seq) for seq in sequences)

# Crear el dataset con relleno
dataset = {}
for i, seq in enumerate(sequences):
    # Convertir a nucleótidos y eliminar tabs (si existen)
    nucleotides = list(seq.replace('\t', ''))

    # Rellenar con '-' si es más corta que la máxima longitud
    nucleotides_filled = list(nucleotides) + ['-'] * (max_length - len(nucleotides))

    # Agregar la clase al final
    nucleotides_filled.append(classes[i])

    dataset[i] = nucleotides_filled

# Crear DataFrame
column_names = [i for i in range(max_length)] + ['Class']
dframe = pd.DataFrame.from_dict(dataset, orient='index', columns=column_names)

# Mostrar las primeras filas
dframe.head()

,0,1,2,3,4,5,6,7,8,9,...,4651,4652,4653,4654,4655,4656,4657,4658,4659,Class
0,M,N,T,H,E,S,E,V,Y,T,...,-,-,-,-,-,-,-,-,-,+
1,M,I,V,L,S,V,G,S,A,S,...,-,-,-,-,-,-,-,-,-,+
2,M,V,R,R,R,P,A,T,G,A,...,-,-,-,-,-,-,-,-,-,+
3,M,L,P,G,L,A,L,L,L,L,...,-,-,-,-,-,-,-,-,-,+
4,M,N,E,P,A,K,H,R,L,G,...,-,-,-,-,-,-,-,-,-,+


In [10]:
dframe.describe()

,0,1,2,3,4,5,6,7,8,9,...,4651,4652,4653,4654,4655,4656,4657,4658,4659,Class
count,246,246,246,246,246,246,246,246,246,246,...,246,246,246,246,246,246,246,246,246,246
unique,3,20,20,20,19,20,20,20,20,20,...,2,2,2,2,2,2,2,2,2,2
top,M,A,S,S,L,S,L,L,L,L,...,-,-,-,-,-,-,-,-,-,+
freq,244,40,40,34,29,38,35,33,31,31,...,245,245,245,245,245,245,245,245,245,123


In [11]:
# describe  doesn't tell us enough information since the attributes are text. Let's record alue counts for each sequence

series = []

for name in dframe.columns:
  series.append(dframe[name].value_counts())

info = pd.DataFrame(series)
details = info.transpose()
details

,count,count,count,count,count,count,count,count,count,count,...,count,count,count,count,count,count,count,count,count,count
M,244.0,3.0,5.0,3.0,1.0,3.0,2.0,2.0,6.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
K,1.0,15.0,8.0,9.0,12.0,8.0,15.0,6.0,5.0,6.0,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
Q,1.0,6.0,16.0,10.0,8.0,15.0,5.0,9.0,7.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
A,NaN,40.0,13.0,17.0,13.0,22.0,27.0,24.0,24.0,16.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
E,NaN,29.0,15.0,13.0,15.0,15.0,17.0,11.0,11.0,9.0,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
S,NaN,23.0,40.0,34.0,18.0,38.0,21.0,20.0,21.0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN
T,NaN,18.0,16.0,14.0,15.0,16.0,6.0,15.0,18.0,20.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
D,NaN,17.0,9.0,3.0,5.0,10.0,6.0,17.0,15.0,6.0,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN
G,NaN,16.0,20.0,24.0,23.0,21.0,12.0,17.0,25.0,19.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
L,NaN,15.0,21.0,27.0,29.0,18.0,35.0,33.0,31.0,31.0,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# Unfortunately, we can't run machine learning algorithms on the data in 'String' formats. As a result, we need to switch it to numerical data.
# This can easily be accomplished using the pd.get_dummies() function
numerical_df = pd.get_dummies(dframe)
numerical_df.head()

,0_K,0_M,0_Q,1_A,1_C,1_D,1_E,1_F,1_G,1_H,...,4656_-,4656_D,4657_-,4657_S,4658_-,4658_D,4659_-,4659_V,Class_+,Class_-
0,False,True,False,False,False,False,False,False,False,False,...,True,False,True,False,True,False,True,False,True,False
1,False,True,False,False,False,False,False,False,False,False,...,True,False,True,False,True,False,True,False,True,False
2,False,True,False,False,False,False,False,False,False,False,...,True,False,True,False,True,False,True,False,True,False
3,False,True,False,False,False,False,False,False,False,False,...,True,False,True,False,True,False,True,False,True,False
4,False,True,False,False,False,False,False,False,False,False,...,True,False,True,False,True,False,True,False,True,False


In [13]:
# We don't need both class columns.  Lets drop one then rename the other to simply 'Class'.
df = numerical_df.drop(columns=['Class_-'])

df.rename(columns = {'Class_+': 'Class'}, inplace = True)
df.head()

,0_K,0_M,0_Q,1_A,1_C,1_D,1_E,1_F,1_G,1_H,...,4655_E,4656_-,4656_D,4657_-,4657_S,4658_-,4658_D,4659_-,4659_V,Class
0,False,True,False,False,False,False,False,False,False,False,...,False,True,False,True,False,True,False,True,False,True
1,False,True,False,False,False,False,False,False,False,False,...,False,True,False,True,False,True,False,True,False,True
2,False,True,False,False,False,False,False,False,False,False,...,False,True,False,True,False,True,False,True,False,True
3,False,True,False,False,False,False,False,False,False,False,...,False,True,False,True,False,True,False,True,False,True
4,False,True,False,False,False,False,False,False,False,False,...,False,True,False,True,False,True,False,True,False,True


In [14]:
# Es importante que justifique la codificación de la secuencias.

# Para poder usar random forest primero se penso en asignarle un valor numerico
# a cada letra del aminoacido pero para evitar asignarle 'mayor peso' a ciertas
# letras mejor se opto por tranformar las secuencias de aminoacidos en vectores
# binarios que mostraran si cierta letra se encuentra o no y en que posicion
# lo hace.

# NOTA: Se consulto Deep Seek para poder llevar acabo la idea.

from sklearn.preprocessing import OneHotEncoder

# 1. Cargar datos (ya tienes combined_df con Columnas: ID, Sequence, Class)
sequences = combined_df['Sequence']

# 2. Crear mapeo de aminoácidos
amino_acids = list('ACDEFGHIKLMNPQRSTVWY')  # 20 aminoácidos estándar
aa_to_int = {aa: i for i, aa in enumerate(amino_acids)}

# 3. One-Hot Encoding
def one_hot_encode(seq, max_length):
    """Codifica una secuencia como one-hot, con padding si es necesario"""
    encoded = np.zeros((max_length, len(amino_acids)))
    for i, aa in enumerate(seq):
        if aa in aa_to_int:
            encoded[i, aa_to_int[aa]] = 1
    return encoded.flatten()  # Convertir a 1D para ML

max_length = sequences.str.len().max()
X = np.array([one_hot_encode(seq, max_length) for seq in sequences])
y = combined_df['Class'].map({'+': 1, '-': 0}).values  # Convertir a binario

print(f"Forma de X: {X.shape}")  # (n_samples, max_length * 20)
X

Forma de X: (246, 93200)


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [28]:
# Use the model_selection module to separate training and testing datasets
from sklearn import model_selection

# Create X and Y datasets for training
#X = np.array(df.drop(['Class'],axis=1))
#y = np.array(df['Class'])

# define seed for reproducibility
seed = 1

# split data into training and testing datasets
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.25, random_state=seed)

In [29]:
# Usando Random Forest
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier(max_depth=5, n_estimators=10, max_features=1)
classifier.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, max_features=1, n_estimators=10)

In [30]:
rfc_pred = classifier.predict(X_test)
print(confusion_matrix(y_test,rfc_pred))
print(classification_report(y_test,rfc_pred))

[[10 18]
 [ 9 25]]
              precision    recall  f1-score   support

           0       0.53      0.36      0.43        28
           1       0.58      0.74      0.65        34

    accuracy                           0.56        62
   macro avg       0.55      0.55      0.54        62
weighted avg       0.56      0.56      0.55        62

